# 🐦‍⬛ BlueMagpie-TTS · Google Colab 試用筆記本

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OpenFormosa/BlueMagpie-TTS/blob/main/notebooks/BlueMagpie_TTS_Colab.ipynb)

**BlueMagpie-TTS** 是針對**台灣華語**與**中英混合（code-switching）**打造的開源文字轉語音（TTS）模型，輸出 48 kHz 語音。這本筆記本帶你在 Colab 上完整跑一遍：

1. 安裝套件、下載模型（約 8 GB，公開模型、不需帳號）
2. **指定語者**合成（內附已授權的語者向量）
3. **聲音複製**（免逐字稿）：快速（單段參考音檔）與穩定（抽語者向量）兩種方式
4. **長文串流**合成

**建議執行階段**：選單「執行階段 → 變更執行階段類型」選 **GPU**（免費的 T4 即可；L4／A100 更快）。CPU 也能跑，只是速度較慢。

> ⚠️ 聲音複製請只使用**你已取得授權**的聲音；合成語音僅供研究與評估，正式使用前請人工檢視。

- 📦 模型：[OpenFormosa/BlueMagpie-TTS](https://huggingface.co/OpenFormosa/BlueMagpie-TTS)
- 💻 程式碼與完整文件：[github.com/OpenFormosa/BlueMagpie-TTS](https://github.com/OpenFormosa/BlueMagpie-TTS)
- 🔊 不想跑程式？線上 demo：[BlueMagpie-TTS Demo Space](https://huggingface.co/spaces/voidful/BlueMagpie-TTS-Demo)

## 1. 安裝

安裝 `bluemagpie`（會自動帶入相依的 `barbet` 套件）與 `soundfile`，約需 1–2 分鐘。

In [ ]:
!pip install -q "git+https://github.com/OpenFormosa/BlueMagpie-TTS.git" soundfile

import bluemagpie
print("bluemagpie ready")

## 2. 下載並載入模型

首次執行會從 Hugging Face 下載約 8 GB 的模型檔（公開模型，**不需要 token**），依 Colab 網速約 2–5 分鐘；載入到 GPU 再約 1–2 分鐘。

In [ ]:
import os
import torch
from huggingface_hub import snapshot_download
from transformers import PreTrainedTokenizerFast
from bluemagpie import BlueMagpieModel

model_dir = snapshot_download("OpenFormosa/BlueMagpie-TTS")

# 直接從 tokenizer.json 載入 tokenizer，相容較新版 transformers（5.x）
tokenizer = PreTrainedTokenizerFast(tokenizer_file=os.path.join(model_dir, "tokenizer.json"))

device = "cuda" if torch.cuda.is_available() else "cpu"
model = BlueMagpieModel.from_local(model_dir, tokenizer=tokenizer, training=False, device=device)

print(f"device={device}, sample_rate={model.sample_rate}")
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## 3. 指定語者合成

用模型內附的**多語者向量表**控制音色。`hung_yi_lee`（李宏毅老師，已取得本人授權）與 `female_voice`（通用女聲）兩個語者可選。

生成參數採用模型卡建議值：`cfg_value=2.0`、`inference_timesteps=10`、`retry_badcase=True`。

In [ ]:
import soundfile as sf
from IPython.display import Audio, display

# 內附語者向量表：{"speaker_ids": [...], "centroids": tensor[N, 192], "dim": 192}
table = torch.load(os.path.join(model_dir, "checkpoints", "speaker_centroids.pt"),
                   map_location="cpu", weights_only=True)
print("內附語者：", table["speaker_ids"])

speaker_id = "hung_yi_lee"   # 改成 "female_voice" 就能換語者
centroid = table["centroids"][table["speaker_ids"].index(speaker_id)]

GEN = dict(cfg_value=2.0, inference_timesteps=10, retry_badcase=True)

text = "你好，這是 BlueMagpie 在 Colab 上合成的第一句話。"
audio = model.generate(target_text=text, speaker_centroid=centroid, **GEN)

wav = audio.squeeze().float().cpu().numpy()
sf.write("sample.wav", wav, model.sample_rate)
display(Audio(wav, rate=model.sample_rate))

中英混合（code-switching）不必改寫就能唸：

In [ ]:
text = "這是 AI TTS code switching 測試，明天的 meeting 改到下午三點。"
audio = model.generate(target_text=text, speaker_centroid=centroid, **GEN)
display(Audio(audio.squeeze().float().cpu().numpy(), rate=model.sample_rate))

## 4. 聲音複製（免逐字稿）· 快速模式

把一段 **3 秒以上的乾淨語音**直接交給模型（`reference_wav_path`，checkpoint `step_0006000` 起正式支援），**不需要**參考音檔的文字稿。

執行下一格會出現上傳按鈕；**若不上傳（按取消）**，就用上一步合成的 `sample.wav` 當參考音檔做示範。

> ⚠️ 請只使用你已取得授權的聲音。

In [ ]:
ref_path = None
try:
    from google.colab import files
    uploaded = files.upload()          # 可上傳 wav / mp3 / m4a
    ref_path = next(iter(uploaded), None)
except ImportError:                     # 不在 Colab（一般 Jupyter）時略過上傳
    pass
if not ref_path:
    ref_path = "sample.wav"
    print("未上傳，改用剛剛合成的 sample.wav 作為參考音檔（示範用）")

text = "今天天氣真好，我們一起出去走走吧。"
audio = model.generate(target_text=text, reference_wav_path=ref_path, **GEN)
display(Audio(audio.squeeze().float().cpu().numpy(), rate=model.sample_rate))

## 5. 聲音複製 · 穩定模式（選用）

品質最穩定的做法：先用 ECAPA-TDNN 從參考音檔抽出 **192 維語者向量**再合成（與內附語者向量同一機制）。多段同一語者的音檔會平均成更穩的音色。需要多裝 `speechbrain`：

In [ ]:
!pip install -q speechbrain

from bluemagpie import extract_speaker_centroid

my_centroid = extract_speaker_centroid(ref_path)     # 也可傳入多段： ["a.wav", "b.wav"]
print("centroid shape:", tuple(my_centroid.shape))

audio = model.generate(target_text="用抽出來的語者向量合成，音色會更穩定。",
                       speaker_centroid=my_centroid, **GEN)
display(Audio(audio.squeeze().float().cpu().numpy(), rate=model.sample_rate))

## 6. 長文合成

長文建議**切句逐句合成**（每句都能套用自動重試）再串接，跨句沿用同一個語者向量以維持音色一致。

> 若需要「邊合成邊播放」的區塊級串流，可改用 `generate_streaming`（產生器，串流模式不支援自動重試），用法見 [README 的〈串流輸出〉](https://github.com/OpenFormosa/BlueMagpie-TTS#串流輸出)。

In [ ]:
import re

long_text = ("台灣藍鵲是台灣特有的鳥類，羽色以亮藍為主，尾羽修長。"
             "牠們常成群在樹林間活動，叫聲響亮而容易辨認。"
             "今天天氣真好，我們一起出去走走，順便看看這些美麗的鳥吧！")

sentences = [s for s in re.split(r"(?<=[。！？])", long_text) if s.strip()]
pieces = []
for sent in sentences:
    audio = model.generate(target_text=sent, speaker_centroid=centroid, **GEN)
    pieces.append(audio.squeeze().float().cpu())
    print("✓", sent)

full = torch.cat(pieces, dim=-1)
display(Audio(full.numpy(), rate=model.sample_rate))

## 下一步

- 完整用法（四種輸入模式、參數說明、批次推論引擎、Apple Silicon MLX 加速）：[GitHub README](https://github.com/OpenFormosa/BlueMagpie-TTS)
- 內部評測數據與解讀：README 的〈效能數據〉章節
- 問題回報與合作：[GitHub Issues](https://github.com/OpenFormosa/BlueMagpie-TTS/issues)

> 模型內附的 `hung_yi_lee` 語者向量已取得本人授權；指定其他語者或聲音複製時，請只使用你已取得授權的聲音。合成語音請勿在未經授權下對外散布。程式碼授權：Apache-2.0。